# Reentrenamiento v3 – Clasificador NTP 900.058
Corrige lo que pasó en v2 (atajo **"foto de celular con mano = Blanco"**) y refuerza **vidrio** y **generales**.

**Qué cambia respecto a v2:**
1. **Parte de v1**, no de v2, para no heredar el atajo.
2. **Tope de fotos de celular por color** (`MAX_CAM_POR_COLOR = 60`), elegidas repartidas entre objetos distintos.
3. **Datasets públicos con fotos reales:** RealWaste (UCI) y la parte `real_world` de *Recyclable and Household Waste* (Kaggle), con tope por color para no desbalancear.
4. **`BLUR_MIN = 20`** (en v2 se descartaron fotos buenas).
5. **Mezcla de pesos v1 + v3** (WiSE-FT): se prueban varias mezclas y se elige **con val**, no con test.
6. Compara **v1, v2 y v3** en el test viejo y en `test_camara` (por foto y por objeto) y solo reemplaza si v3 gana.

**Antes de correr:** pasa a mano las **envolturas metalizadas** (snacks) de `data basurita/Blanco` y `test_camara/Blanco` a la carpeta **Negro**.

**Entorno:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`. Tarda ~40–60 min (incluye descargar ~1.5 GB de datasets).

## 1. Instalar y verificar la GPU

In [ ]:
!pip install -q "ultralytics>=8.3,<8.4" imagehash kagglehub
import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Conectar Drive y configuración

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, re, shutil, zipfile, random
from collections import Counter, defaultdict
from datetime import datetime

DRIVE_DIR = "/content/drive/MyDrive/data basurita"
ZIP_V1 = f"{DRIVE_DIR}/dataset_ntp900058.zip"          # particion vieja (congelada)
MODELOS = f"{DRIVE_DIR}/modelos"
MODELO_V1 = f"{MODELOS}/clasificador_v1.pt"              # el original
MODELO_V2 = f"{MODELOS}/clasificador_v2.pt"              # el del atajo (solo para comparar)
RUNS = f"{DRIVE_DIR}/runs_clasificador"
DATASET = "/content/dataset_v3"
CAMARA = "/content/test_camara"
REVISION = f"{DRIVE_DIR}/revision_v3"

SEMILLA = 42
PCT_VAL = 0.15               # de los datos NUEVOS, cuanto va a val
PCT_TEST_CAM = 0.15          # (solo si hay ordenadas_camara nuevas)
GAP_SEG = 60                 # fotos con mas de 60 s de diferencia = otro objeto
MAX_CAM_POR_COLOR = 60       # tope de fotos de celular por color en train+val
MAX_PUBLICO_POR_COLOR = 400  # tope de fotos de datasets publicos por color
SOLO_REAL_WORLD = True       # del dataset de Kaggle, usar solo fotos en entornos reales
LADO_MAX = 512
EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
random.seed(SEMILLA)

NOMBRES = {
    "Azul": "azul_papel_carton",
    "Amarillo": "amarillo_metales",
    "Blanco": "blanco_plasticos",
    "Gris": "gris_vidrio",
    "Marron": "marron_organicos",
    "Rojo": "rojo_peligrosos",
    "Negro": "negro_generales",
}
LARGO_A_CORTO = {v: k for k, v in NOMBRES.items()}

def imagenes(carpeta):
    if not os.path.isdir(carpeta):
        return []
    return sorted(f for f in os.listdir(carpeta) if f.lower().endswith(EXT))

def hora(nombre):
    m = re.search(r"(\d{8}_\d{6})", nombre)
    return datetime.strptime(m.group(1), "%Y%m%d_%H%M%S") if m else None

def agrupar_por_hora(fotos):
    fotos = sorted(fotos, key=lambda f: (hora(f) or datetime.min, f))
    grupos, prev = [], None
    for f in fotos:
        t = hora(f)
        if prev is None or t is None or (t - prev).total_seconds() > GAP_SEG:
            grupos.append([])
        grupos[-1].append(f)
        prev = t
    return grupos

for ruta in (MODELO_V1, ZIP_V1):
    assert os.path.exists(ruta), f"No encuentro {ruta}"
print("v2 disponible para comparar:", os.path.exists(MODELO_V2))

## 3. Pasar `ordenadas_camara` a su lugar (si tomaste fotos nuevas)
Igual que en v2: objetos completos a `test_camara` y el resto a entrenamiento con prefijo `cam_`. Si no hay fotos nuevas, no hace nada.

In [ ]:
ORDENADAS = f"{DRIVE_DIR}/ordenadas_camara"
movidas = 0
if os.path.isdir(ORDENADAS):
    for corto, largo in NOMBRES.items():
        origen = f"{ORDENADAS}/{largo}"
        fotos = imagenes(origen)
        if not fotos:
            continue
        grupos = agrupar_por_hora(fotos)
        random.shuffle(grupos)
        n_test = max(1, round(len(grupos) * PCT_TEST_CAM))
        for i, g in enumerate(grupos):
            destino = f"{DRIVE_DIR}/test_camara/{corto}" if i < n_test else f"{DRIVE_DIR}/{corto}"
            os.makedirs(destino, exist_ok=True)
            for f in g:
                shutil.move(f"{origen}/{f}", f"{destino}/cam_{f}")
                movidas += 1
        print(f"{corto:9} objetos={len(grupos):3}  a test_camara={n_test}")
print("Fotos movidas:", movidas)

## 4. Cargar la partición vieja (congelada)

In [ ]:
shutil.rmtree(DATASET, ignore_errors=True)
zipfile.ZipFile(ZIP_V1).extractall(DATASET)
for split in ("train", "val", "test"):
    for corto, largo in NOMBRES.items():
        if os.path.isdir(f"{DATASET}/{split}/{corto}"):
            os.rename(f"{DATASET}/{split}/{corto}", f"{DATASET}/{split}/{largo}")

def conteo(raiz, splits=("train", "val", "test")):
    print(f"{'clase':<20}" + "".join(f"{s:>8}" for s in splits))
    for c in NOMBRES.values():
        print(f"{c:<20}" + "".join(f"{len(imagenes(f'{raiz}/{s}/{c}')):>8}" for s in splits))

print("Particion vieja (v1):")
conteo(DATASET)

## 5. Copiar `test_camara`
Nunca se usa para entrenar ni para elegir el modelo; solo para el reporte final.

In [ ]:
shutil.rmtree(CAMARA, ignore_errors=True)
for corto, largo in NOMBRES.items():
    origen = f"{DRIVE_DIR}/test_camara/{corto}"
    os.makedirs(f"{CAMARA}/{largo}", exist_ok=True)
    for f in imagenes(origen):
        shutil.copy2(f"{origen}/{f}", f"{CAMARA}/{largo}/{f}")
    n = len(imagenes(f"{CAMARA}/{largo}"))
    print(f"{largo:<20}{n:>5}" + ("   <-- AVISO: vacio" if n == 0 else "   <-- AVISO: muy pocas" if n < 5 else ""))

## 6. Descargar datasets públicos con fotos reales
| Dataset | Qué aporta | Mapeo a colores |
|---|---|---|
| **RealWaste** (UCI, CC BY 4.0) | fotos reales de relleno sanitario | Glass→Gris, Miscellaneous Trash→Negro, Food Organics/Vegetation→Marrón, Cardboard/Paper→Azul, Metal→Amarillo, Plastic→Blanco (Textile se ignora) |
| **Recyclable and Household Waste** (Kaggle) | 30 categorías, carpeta `real_world` | glass_*→Gris, styrofoam_* y paper_cups→Negro, aerosol_cans→Rojo, latas→Amarillo, cartón/papel→Azul, comida/café/cáscaras/té→Marrón, botellas y envases plásticos→Blanco |

Se imprime qué categoría fue a qué color: **revísalo**; lo que sale como `ignorada` no se usa.

In [ ]:
import kagglehub, urllib.request, glob

# --- Recyclable and Household Waste (Kaggle) ---
RUTA_HH = kagglehub.dataset_download("alistairking/recyclable-and-household-waste-classification")
print("Kaggle en:", RUTA_HH)

# --- RealWaste (UCI) ---
RW_DIR = "/content/realwaste"
if not os.path.isdir(RW_DIR):
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/908/realwaste.zip", "/content/realwaste.zip")
    zipfile.ZipFile("/content/realwaste.zip").extractall(RW_DIR)
    for z in glob.glob(f"{RW_DIR}/**/*.zip", recursive=True):   # por si viene un zip dentro de otro
        zipfile.ZipFile(z).extractall(os.path.dirname(z))
print("RealWaste en:", RW_DIR)

REGLAS_HH = [  # (texto en el nombre de la categoria, color) - el orden importa
    ("aerosol", "Rojo"), ("glass", "Gris"), ("styrofoam", "Negro"), ("paper_cup", "Negro"),
    ("_cans", "Amarillo"), ("cardboard", "Azul"), ("newspaper", "Azul"), ("magazine", "Azul"),
    ("office_paper", "Azul"), ("food_waste", "Marron"), ("coffee", "Marron"), ("eggshell", "Marron"),
    ("tea_bag", "Marron"), ("plastic_water", "Blanco"), ("plastic_soda", "Blanco"),
    ("plastic_detergent", "Blanco"), ("plastic_food_container", "Blanco"), ("plastic_cup_lid", "Blanco"),
    ("cutlery", "Blanco"),
]
MAPEO_RW = {"cardboard": "Azul", "paper": "Azul", "food organics": "Marron", "vegetation": "Marron",
            "glass": "Gris", "metal": "Amarillo", "plastic": "Blanco", "miscellaneous trash": "Negro"}

def color_hh(cat):
    for clave, color in REGLAS_HH:
        if clave in cat.lower():
            return color
    return None

publicos = defaultdict(lambda: defaultdict(list))   # color -> fuente -> [rutas]
asignacion = Counter()
for raiz, _, archivos in os.walk(RUTA_HH):
    fotos = [f for f in archivos if f.lower().endswith(EXT)]
    if not fotos:
        continue
    base = os.path.basename(raiz).lower()
    if base in ("default", "real_world"):
        if SOLO_REAL_WORLD and base != "real_world":
            continue
        cat = os.path.basename(os.path.dirname(raiz))
    else:
        cat = os.path.basename(raiz)
    color = color_hh(cat)
    asignacion[("kaggle", cat, color or "ignorada")] += len(fotos)
    if color:
        publicos[color][f"hh/{cat}"] += [os.path.join(raiz, f) for f in fotos]

for raiz, _, archivos in os.walk(RW_DIR):
    fotos = [f for f in archivos if f.lower().endswith(EXT)]
    if not fotos:
        continue
    cat = os.path.basename(raiz)
    color = MAPEO_RW.get(cat.strip().lower())
    asignacion[("realwaste", cat, color or "ignorada")] += len(fotos)
    if color:
        publicos[color][f"rw/{cat}"] += [os.path.join(raiz, f) for f in fotos]

print(f"\n{'fuente':<10}{'categoria':<32}{'color':<10}{'fotos':>6}")
for (fuente, cat, color), n in sorted(asignacion.items()):
    print(f"{fuente:<10}{cat:<32}{color:<10}{n:>6}")

def elegir_balanceado(por_fuente, tope):
    # se turna entre categorias para que ningun tipo de objeto domine el color
    grupos = [random.sample(l, len(l)) for l in por_fuente.values()]
    elegidas = []
    while len(elegidas) < tope and any(grupos):
        for g in grupos:
            if g and len(elegidas) < tope:
                elegidas.append(g.pop())
    return elegidas

PUBLICO_ELEGIDO = {c: elegir_balanceado(publicos[c], MAX_PUBLICO_POR_COLOR) for c in NOMBRES}
print("\nFotos publicas candidatas por color (antes del ETL):",
      {c: len(v) for c, v in PUBLICO_ELEGIDO.items()})

## 7. ETL – limpieza de todos los datos nuevos
| Etapa | Qué se hace |
|---|---|
| **Extract** | Fotos de `data basurita/<Color>` (internet + `cam_`) y de los datasets públicos elegidos. |
| **Transform** | Se descartan ilegibles, miniaturas, borrosas (`BLUR_MIN = 20`), las que ya estaban en v1 y las casi iguales a test. Luego se aplica el **tope de fotos de celular por color**, repartido entre objetos distintos. |
| **Load** | Paso 8: se reducen a 512 px y se reparten en train/val por objeto. |

In [ ]:
import numpy as np, imagehash, cv2
import pandas as pd
from PIL import Image, ImageOps

MIN_LADO = 128
MIN_KB = 5
BLUR_MIN = 20

def huella(ruta):
    try:
        with Image.open(ruta) as im:
            return int(str(imagehash.phash(ImageOps.exif_transpose(im).convert("RGB"))), 16)
    except Exception:
        return None

def inspeccionar(ruta):
    try:
        with Image.open(ruta) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            h_p = str(imagehash.phash(im))
            im.thumbnail((LADO_MAX, LADO_MAX))
            gris = cv2.cvtColor(np.array(im), cv2.COLOR_RGB2GRAY)
            nitidez = float(cv2.Laplacian(gris, cv2.CV_64F).var())
        return {"ancho": w, "alto": h, "kb": os.path.getsize(ruta) / 1024, "nitidez": nitidez, "huella": h_p}
    except Exception:
        return None

def huellas(rutas):
    hs = [huella(r) for r in rutas]
    return np.array([h for h in hs if h is not None], dtype=np.uint64)

def distancia_min(h, arr):
    if len(arr) == 0:
        return 64
    x = np.bitwise_xor(arr, np.uint64(h))
    return int(np.unpackbits(x.view(np.uint8)).reshape(-1, 64).sum(1).min())

viejas_todas = [f"{DATASET}/{s}/{c}/{f}" for s in ("train", "val", "test")
                for c in NOMBRES.values() for f in imagenes(f"{DATASET}/{s}/{c}")]
de_test = [f"{DATASET}/test/{c}/{f}" for c in NOMBRES.values() for f in imagenes(f"{DATASET}/test/{c}")]
de_test += [f"{CAMARA}/{c}/{f}" for c in NOMBRES.values() for f in imagenes(f"{CAMARA}/{c}")]
print("Calculando huellas de lo viejo (unos minutos)...")
H_VIEJAS = huellas(viejas_todas)
H_TEST = huellas(de_test)
print(f"viejas: {len(H_VIEJAS)} | test (viejo + camara): {len(H_TEST)}")

candidatas = []   # (ruta, clase_larga, fuente)
for corto, largo in NOMBRES.items():
    for f in imagenes(f"{DRIVE_DIR}/{corto}"):
        candidatas.append((f"{DRIVE_DIR}/{corto}/{f}", largo, "cam" if f.startswith("cam_") else "web"))
    for ruta in PUBLICO_ELEGIDO[corto]:
        candidatas.append((ruta, largo, "publico"))

registros = []
for ruta, largo, fuente in candidatas:
    info = inspeccionar(ruta)
    if info is None:
        registros.append({"ruta": ruta, "clase": largo, "fuente": fuente, "estado": "ilegible"})
        continue
    hh = int(info["huella"], 16)
    if min(info["ancho"], info["alto"]) < MIN_LADO or (fuente != "publico" and info["kb"] < MIN_KB):
        estado = "pequena"
    elif distancia_min(hh, H_VIEJAS) <= 4:
        estado = "ya_en_v1"
    elif distancia_min(hh, H_TEST) <= 8:
        estado = "parecida_a_test"
    elif info["nitidez"] < BLUR_MIN:
        estado = "borrosa"
    else:
        estado = "nueva"
    registros.append({"ruta": ruta, "clase": largo, "fuente": fuente, "estado": estado, **info})
ETL = pd.DataFrame(registros)

# Tope de fotos de celular por color, turnandose entre objetos (grupos por hora)
for largo in NOMBRES.values():
    sel = ETL[(ETL.clase == largo) & (ETL.fuente == "cam") & (ETL.estado == "nueva")]
    if len(sel) <= MAX_CAM_POR_COLOR:
        continue
    por_nombre = {os.path.basename(r): i for i, r in zip(sel.index, sel.ruta)}
    grupos = [random.sample(g, len(g)) for g in agrupar_por_hora(list(por_nombre))]
    elegidas = set()
    while len(elegidas) < MAX_CAM_POR_COLOR:
        for g in grupos:
            if g and len(elegidas) < MAX_CAM_POR_COLOR:
                elegidas.add(por_nombre[g.pop()])
    ETL.loc[sel.index.difference(list(elegidas)), "estado"] = "sobra_tope_cam"

print("Reporte ETL (fotos por estado):")
print(pd.crosstab([ETL.clase, ETL.fuente], ETL.estado, margins=True, margins_name="TOTAL"))

nuevas = defaultdict(list)
for r in ETL[ETL.estado == "nueva"].itertuples():
    nuevas[r.clase].append((r.ruta, int(r.huella, 16), r.fuente))

## 8. Partir los datos nuevos en train/val (85/15) y juntarlos con lo viejo
- Celular: agrupadas **por objeto** (hora del nombre).
- Internet y públicas: las casi iguales entre sí van juntas.

In [ ]:
def grupos_por_huella(items):
    padre = list(range(len(items)))
    def raiz(i):
        while padre[i] != i:
            padre[i] = padre[padre[i]]; i = padre[i]
        return i
    hs = np.array([h for _, h, _ in items], dtype=np.uint64)
    for i in range(len(items)):
        x = np.bitwise_xor(hs[i + 1:], hs[i])
        if len(x):
            d = np.unpackbits(x.view(np.uint8)).reshape(-1, 64).sum(1)
            for j in np.nonzero(d <= 6)[0]:
                padre[raiz(i + 1 + j)] = raiz(i)
    g = defaultdict(list)
    for i, it in enumerate(items):
        g[raiz(i)].append(it)
    return list(g.values())

def guardar(origen, destino):
    with Image.open(origen) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.thumbnail((LADO_MAX, LADO_MAX))
        im.save(destino, "JPEG", quality=90)

agregadas = defaultdict(Counter)
for largo, items in nuevas.items():
    cam = [it for it in items if it[2] == "cam"]
    otras = [it for it in items if it[2] != "cam"]
    por_nombre = {os.path.basename(r): (r, h, s) for r, h, s in cam}
    grupos = [[por_nombre[f] for f in g] for g in agrupar_por_hora(list(por_nombre))] + grupos_por_huella(otras)
    random.shuffle(grupos)
    meta_val, en_val = PCT_VAL * len(items), 0
    for k_g, g in enumerate(grupos):
        split = "val" if en_val < meta_val else "train"
        if split == "val":
            en_val += len(g)
        os.makedirs(f"{DATASET}/{split}/{largo}", exist_ok=True)
        for k, (ruta, _, fuente) in enumerate(g):
            base = re.sub(r"[^\w.-]", "_", os.path.splitext(os.path.basename(ruta))[0])
            nombre = f"v3_{fuente}_{k_g}_{k}_{base}.jpg"
            try:
                guardar(ruta, f"{DATASET}/{split}/{largo}/{nombre}")
                agregadas[largo][f"{split}_{fuente}"] += 1
            except Exception as e:
                print("omitida:", ruta, e)

print(pd.DataFrame(agregadas).T.fillna(0).astype(int))
print("\nDataset v3 final (viejo + nuevo). El test es el viejo, sin cambios:")
conteo(DATASET)

## 9. Pre-EDA – revisar los datos ANTES de entrenar

In [ ]:
import matplotlib.pyplot as plt

def fuente_de(nombre):
    for pref, etiqueta in (("v3_cam", "celular"), ("v3_web", "internet"), ("v3_publico", "publico")):
        if nombre.startswith(pref):
            return etiqueta
    return "v1 (original)"

filas = []
for split in ("train", "val", "test"):
    for c in NOMBRES.values():
        for f in imagenes(f"{DATASET}/{split}/{c}"):
            filas.append({"split": split, "clase": c, "fuente": fuente_de(f)})
for c in NOMBRES.values():
    for f in imagenes(f"{CAMARA}/{c}"):
        filas.append({"split": "test_camara", "clase": c, "fuente": "celular"})
INVENTARIO = pd.DataFrame(filas)

print("Fotos por color y split:")
print(pd.crosstab(INVENTARIO.clase, INVENTARIO.split, margins=True, margins_name="TOTAL")
        [["train", "val", "test", "test_camara", "TOTAL"]])

tr = pd.crosstab(INVENTARIO[INVENTARIO.split == "train"].clase, INVENTARIO[INVENTARIO.split == "train"].fuente)
print("\nTrain por fuente (lo clave: 'celular' parejo entre colores):")
print(tr)
ax = tr.plot.barh(stacked=True, figsize=(9, 4))
ax.set_title("Train por color y fuente"); ax.set_xlabel("fotos"); plt.tight_layout(); plt.show()
print(f"Desbalance total en train: {tr.sum(axis=1).max() / tr.sum(axis=1).min():.1f}x")
if "celular" in tr:
    cel = tr["celular"].replace(0, np.nan)
    print(f"Desbalance de fotos de celular: {cel.max() / cel.min():.1f}x (en v2 era ~25x)")

ok = ETL.dropna(subset=["nitidez"])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
a1.hist(np.log10(ok.nitidez + 1), bins=40)
a1.axvline(np.log10(BLUR_MIN + 1), color="red", ls="--", label=f"BLUR_MIN={BLUR_MIN}")
a1.set_title("Nitidez de las fotos nuevas (log10)"); a1.legend()
a2.hist(ok[["ancho", "alto"]].min(axis=1), bins=40)
a2.axvline(MIN_LADO, color="red", ls="--", label=f"MIN_LADO={MIN_LADO}")
a2.set_title("Lado menor (px)"); a2.legend()
plt.tight_layout(); plt.show()

def galeria(rutas, titulos, titulo, n=12):
    rutas, titulos = list(rutas)[:n], list(titulos)[:n]
    if not rutas:
        print(titulo, "-> (nada)"); return
    cols = 6; filas_g = (len(rutas) + cols - 1) // cols
    fig, ejes = plt.subplots(filas_g, cols, figsize=(2.3 * cols, 2.5 * filas_g), squeeze=False)
    for ax in ejes.flat: ax.axis("off")
    for ax, r, t in zip(ejes.flat, rutas, titulos):
        with Image.open(r) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert("RGB"))
        ax.set_title(t, fontsize=8)
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

bor = ETL[ETL.estado == "borrosa"].sort_values("nitidez", ascending=False)
galeria(bor.ruta, [f"{c[:10]} n={v:.0f}" for c, v in zip(bor.clase, bor.nitidez)],
        "Descartadas por BORROSAS (las mas nitidas primero)")
pub = ETL[(ETL.estado == "nueva") & (ETL.fuente == "publico")].groupby("clase").head(2)
galeria(pub.ruta, [LARGO_A_CORTO[c] for c in pub.clase], "Muestra de fotos publicas aceptadas (revisa el mapeo)", n=14)

## 10. Reentrenar desde **v1**
Mismo esquema que antes (SGD, aprendizaje bajo, early stopping con `patience=6`; `best.pt` se elige con val).

In [ ]:
from ultralytics import YOLO

YOLO(MODELO_V1).train(
    data=DATASET,
    epochs=20,
    imgsz=224,
    batch=64,
    optimizer="SGD",
    lr0=0.001,
    lrf=0.1,
    warmup_epochs=0,
    patience=6,
    seed=SEMILLA,
    project=RUNS,
    name="v3_ajuste",
    exist_ok=True,
)
MODELO_V3_PURO = f"{RUNS}/v3_ajuste/weights/best.pt"
# Si Colab se desconecto: YOLO(f"{RUNS}/v3_ajuste/weights/last.pt").train(resume=True)

## 11. Mezcla de pesos v1 + v3 y elección **con val**
Se promedian los pesos: `alpha = 1` es v3 puro y `alpha = 0.5` es mitad v1, mitad v3.
La mezcla suele conservar lo que v1 hacía bien y sumar lo nuevo de v3.
**Se elige con val** (promedio entre val viejo y val nuevo), así `test_camara` sigue siendo un examen honesto.

In [ ]:
def evaluar(ruta_modelo, carpeta):
    m = YOLO(ruta_modelo)
    filas = []
    for clase in sorted(os.listdir(carpeta)):
        fotos = [f"{carpeta}/{clase}/{f}" for f in imagenes(f"{carpeta}/{clase}")]
        for ruta, r in zip(fotos, m.predict(fotos, imgsz=224, verbose=False, stream=True)):
            filas.append({"ruta": ruta, "real": clase, "pred": m.names[int(r.probs.top1)],
                          "conf": float(r.probs.top1conf)})
    df = pd.DataFrame(filas)
    df["ok"] = df.real == df.pred
    return df

def mezclar(ruta_a, ruta_b, alpha, salida):
    ck_a = torch.load(ruta_a, map_location="cpu", weights_only=False)
    ck_b = torch.load(ruta_b, map_location="cpu", weights_only=False)
    m_a = (ck_a["ema"] if ck_a.get("ema") is not None else ck_a["model"]).float()
    m_b = (ck_b["ema"] if ck_b.get("ema") is not None else ck_b["model"]).float()
    sd_a, sd_b = m_a.state_dict(), m_b.state_dict()
    mezcla = {k: (alpha * sd_b[k] + (1 - alpha) * sd_a[k]) if sd_a[k].is_floating_point() else sd_b[k]
              for k in sd_a}
    m_a.load_state_dict(mezcla)
    ck_a["model"], ck_a["ema"] = m_a.half(), None
    torch.save(ck_a, salida)
    return salida

CANDIDATOS = {"v3_puro": MODELO_V3_PURO}
for alpha in (0.5, 0.7):
    CANDIDATOS[f"mezcla_{alpha}"] = mezclar(MODELO_V1, MODELO_V3_PURO, alpha, f"/content/mezcla_{alpha}.pt")
CANDIDATOS["v1"] = MODELO_V1

puntaje = {}
for nombre, ruta in CANDIDATOS.items():
    df = evaluar(ruta, f"{DATASET}/val")
    nuevo = df.ruta.map(lambda r: os.path.basename(r).startswith("v3_"))
    acc_viejo, acc_nuevo = df[~nuevo].ok.mean(), df[nuevo].ok.mean()
    puntaje[nombre] = (acc_viejo + acc_nuevo) / 2
    print(f"{nombre:<12} val viejo {acc_viejo:.1%} | val nuevo {acc_nuevo:.1%} | puntaje {puntaje[nombre]:.1%}")

ELEGIDO = max((k for k in puntaje if k != "v1"), key=puntaje.get)
MODELO_V3 = CANDIDATOS[ELEGIDO]
print("\nElegido como v3 (por val):", ELEGIDO)

## 12. Comparar v1, v2 y v3 en los tests
> Mira `test_camara` **una sola vez**. Con pocos objetos, revisa sobre todo el resultado **por objeto**.

In [ ]:
MODELOS_CMP = {"v1": MODELO_V1, "v3": MODELO_V3}
if os.path.exists(MODELO_V2):
    MODELOS_CMP = {"v1": MODELO_V1, "v2": MODELO_V2, "v3": MODELO_V3}

R = {(v, p): evaluar(ruta, carpeta)
     for v, ruta in MODELOS_CMP.items()
     for p, carpeta in (("test_viejo", f"{DATASET}/test"), ("test_camara", CAMARA))}
ACC = {k: df.ok.mean() for k, df in R.items()}

def por_objeto(df):
    df = df.assign(nombre=df.ruta.map(os.path.basename))
    filas = []
    for clase, g in df.groupby("real"):
        for grupo in agrupar_por_hora(list(g.nombre)):
            sub = g[g.nombre.isin(grupo)]
            filas.append({"clase": clase, "ok": sub.pred.mode()[0] == clase})
    return pd.DataFrame(filas)
OBJ = {v: por_objeto(R[(v, "test_camara")]) for v in MODELOS_CMP}

print(f"{'':<22}" + "".join(f"{v:>9}" for v in MODELOS_CMP))
for p in ("test_viejo", "test_camara"):
    print(f"{p + ' (por foto)':<22}" + "".join(f"{ACC[(v, p)]:>9.1%}" for v in MODELOS_CMP))
print(f"{'test_camara (objetos)':<22}" + "".join(f"{f'{OBJ[v].ok.sum()}/{len(OBJ[v])}':>9}" for v in MODELOS_CMP))

print("\nAcierto por color en test_camara:")
tabla = pd.DataFrame({"fotos": R[("v3", "test_camara")].groupby("real").size(),
                      **{v: R[(v, "test_camara")].groupby("real").ok.mean() for v in MODELOS_CMP}})
print(tabla.to_string(formatters={v: "{:.0%}".format for v in MODELOS_CMP}))

print("\nA donde manda cada modelo los errores de test_camara (para detectar atajos):")
for v in MODELOS_CMP:
    df = R[(v, "test_camara")]
    print(f"  {v}: { {k: int(n) for k, n in df[~df.ok].pred.map(LARGO_A_CORTO).value_counts().items()} }")

## 13. Post-EDA de v3

In [ ]:
from IPython.display import Image as IPImage, display

curvas = f"{RUNS}/v3_ajuste/results.png"
if os.path.exists(curvas):
    display(IPImage(curvas, width=900))
hist = pd.read_csv(f"{RUNS}/v3_ajuste/results.csv")
hist.columns = [c.strip() for c in hist.columns]
col_acc = [c for c in hist.columns if "top1" in c][0]
print(f"Epocas: {len(hist)} | mejor val top-1: {hist[col_acc].max():.3f} (epoca {int(hist[col_acc].idxmax()) + 1})")

clases = list(NOMBRES.values()); cortos = [LARGO_A_CORTO[c] for c in clases]
fig, ejes = plt.subplots(1, len(MODELOS_CMP), figsize=(7 * len(MODELOS_CMP), 5.5))
for ax, v in zip(np.atleast_1d(ejes), MODELOS_CMP):
    df = R[(v, "test_camara")]
    cm = pd.crosstab(df.real, df.pred, normalize="index").reindex(index=clases, columns=clases, fill_value=0)
    ax.imshow(cm.values, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(clases)):
        for j in range(len(clases)):
            if cm.values[i, j] >= 0.01:
                ax.text(j, i, f"{cm.values[i, j]:.0%}", ha="center", va="center",
                        color="white" if cm.values[i, j] > 0.5 else "black", fontsize=8)
    ax.set_xticks(range(len(clases)), cortos, rotation=45); ax.set_yticks(range(len(clases)), cortos)
    ax.set_xlabel("prediccion"); ax.set_ylabel("real"); ax.set_title(f"{v} – test_camara ({ACC[(v, 'test_camara')]:.0%})")
plt.tight_layout(); plt.show()

df3 = R[("v3", "test_camara")]
plt.figure(figsize=(7, 3.2))
plt.hist(df3[df3.ok].conf, bins=20, range=(0, 1), alpha=0.7, label="aciertos", color="#2e9e5b")
plt.hist(df3[~df3.ok].conf, bins=20, range=(0, 1), alpha=0.7, label="errores", color="#d64545")
plt.title("v3: confianza en test_camara"); plt.xlabel("confianza"); plt.legend(); plt.tight_layout(); plt.show()
for u in (0.55, 0.65, 0.75):
    s = df3[df3.conf >= u]
    print(f"umbral {u:.2f}: decide en {len(s) / len(df3):.0%} de fotos, con acierto {s.ok.mean():.0%}")

err = df3[~df3.ok].sort_values("conf", ascending=False)
galeria(err.ruta, [f"{LARGO_A_CORTO[r]} -> {LARGO_A_CORTO[p]} {c:.0%}" for r, p, c in zip(err.real, err.pred, err.conf)],
        "Errores de v3 en test_camara (real -> predicho, confianza)")

## 14. Decidir y guardar
**v3 reemplaza a `clasificador.pt` solo si**, frente a v1: no baja en `test_camara` (ni por foto ni por objeto) y no baja más de 2 puntos en el test viejo.

In [ ]:
ok_cam = ACC[("v3", "test_camara")] >= ACC[("v1", "test_camara")]
ok_obj = OBJ["v3"].ok.sum() >= OBJ["v1"].ok.sum()
ok_viejo = ACC[("v3", "test_viejo")] >= ACC[("v1", "test_viejo")] - 0.02
GANA_V3 = ok_cam and ok_obj and ok_viejo

shutil.copy(MODELO_V3, f"{MODELOS}/clasificador_v3.pt")
if GANA_V3:
    shutil.copy(MODELO_V3, f"{MODELOS}/clasificador.pt")
    print(f"v3 ({ELEGIDO}) GANA -> modelos/clasificador.pt ahora es v3")
else:
    shutil.copy(MODELO_V1, f"{MODELOS}/clasificador.pt")
    print(f"v3 NO gana (camara={ok_cam}, objetos={ok_obj}, test viejo={ok_viejo}) -> clasificador.pt vuelve a v1")
print("Guardados: clasificador_v1.pt, clasificador_v2.pt, clasificador_v3.pt")

os.makedirs(REVISION, exist_ok=True)
ETL.drop(columns=["huella"], errors="ignore").to_csv(f"{REVISION}/reporte_etl.csv", index=False)
INVENTARIO.to_csv(f"{REVISION}/inventario_splits.csv", index=False)
for (v, p), df in R.items():
    df.to_csv(f"{REVISION}/predicciones_{v}_{p}.csv", index=False)
pd.Series(puntaje).to_csv(f"{REVISION}/eleccion_por_val.csv")
zip_v3 = shutil.make_archive("/content/dataset_ntp900058_v3", "zip", DATASET)
shutil.copy(zip_v3, f"{DRIVE_DIR}/dataset_ntp900058_v3.zip")
print("Evidencias en:", REVISION)

## 15. Descargar el modelo para tu Mac

In [ ]:
from google.colab import files
shutil.copy(f"{MODELOS}/clasificador.pt", "/content/clasificador.pt")
files.download("/content/clasificador.pt")